# Reproduce Results

This notebook **loads pre-trained models from disk** and evaluates them on the
train / validation / test splits.  No training happens here.

Metrics reported:
- **ROC-AUC** (`sklearn.metrics.roc_auc_score`)
- **Precision** (`sklearn.metrics.precision_score`)
- **Recall** (`sklearn.metrics.recall_score`)
- **F1** (`sklearn.metrics.f1_score`)

In [15]:
import os
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split

import utils
from utils import (
    load_object,
    get_combined_features,
    evaluate_model,
    get_features_from_df
)

ImportError: cannot import name 'get_features_from_df' from 'utils' (/home/arnau/Master Data Science/NLP/NLP_quora_challenge/utils.py)

## 1. Data loading and splitting

The exact same random seed and split ratios as `train_models.ipynb` are used
so that train / val / test sets are identical across notebooks.

In [9]:
DATA_PATH = os.path.expanduser("~/Datasets/QuoraQuestionPairs/quora_data.csv")
quora_df = pd.read_csv(DATA_PATH)

A_df, test_df = train_test_split(quora_df, test_size=0.05, random_state=123)
train_df, val_df = train_test_split(A_df,  test_size=0.05, random_state=123)

print(f'train_df.shape = {train_df.shape}')
print(f'val_df.shape   = {val_df.shape}')
print(f'test_df.shape  = {test_df.shape}')

y_train = train_df["is_duplicate"].values
y_val   = val_df["is_duplicate"].values
y_test  = test_df["is_duplicate"].values

train_df.shape = (291897, 6)
val_df.shape   = (15363, 6)
test_df.shape  = (16172, 6)


## 2. Load models and vectorizers

In [10]:
MODELS_DIR = "models"

count_vectorizer    = load_object(os.path.join(MODELS_DIR, "count_vectorizer.pkl"))
tfidf_vectorizer    = load_object(os.path.join(MODELS_DIR, "tfidf_vectorizer.pkl"))
baseline_logistic   = load_object(os.path.join(MODELS_DIR, "baseline_logistic.pkl"))
improved_logistic   = load_object(os.path.join(MODELS_DIR, "improved_logistic.pkl"))

print("All models and vectorizers loaded successfully.")

All models and vectorizers loaded successfully.


## 3. Feature extraction

- **Baseline**: sparse BoW matrix (CountVectorizer, unigrams)
- **Improved**: BoW + 5 handcrafted similarity features (see `utils.py`)

In [11]:
print("Extracting baseline (BoW) features...")
X_train_bow = utils.get_features_from_df(train_df, count_vectorizer)
X_val_bow   = utils.get_features_from_df(val_df,   count_vectorizer)
X_test_bow  = utils.get_features_from_df(test_df,  count_vectorizer)
print(f"  train shape: {X_train_bow.shape}")

print("\nExtracting combined (BoW + handcrafted) features...")
X_train_comb = get_combined_features(train_df, count_vectorizer, tfidf_vectorizer)
X_val_comb   = get_combined_features(val_df,   count_vectorizer, tfidf_vectorizer)
X_test_comb  = get_combined_features(test_df,  count_vectorizer, tfidf_vectorizer)
print(f"  train shape: {X_train_comb.shape}")

Extracting baseline (BoW) features...


AttributeError: module 'utils' has no attribute 'get_features_from_df'

## 4. Evaluation — ROC-AUC, Precision, Recall, F1

In [ ]:
rows = []

# Baseline model on all three splits
rows.append(evaluate_model(baseline_logistic, X_train_bow, y_train, "baseline", "train"))
rows.append(evaluate_model(baseline_logistic, X_val_bow,   y_val,   "baseline", "val"))
rows.append(evaluate_model(baseline_logistic, X_test_bow,  y_test,  "baseline", "test"))

# Improved model on all three splits
rows.append(evaluate_model(improved_logistic, X_train_comb, y_train, "improved", "train"))
rows.append(evaluate_model(improved_logistic, X_val_comb,   y_val,   "improved", "val"))
rows.append(evaluate_model(improved_logistic, X_test_comb,  y_test,  "improved", "test"))

results_df = pd.DataFrame(rows)
results_df = results_df.set_index(["model", "split"])
display(results_df)

## 5. Summary pivot

In [ ]:
pivot = results_df.reset_index().pivot(index="split", columns="model")
# Reorder rows: train → val → test
pivot = pivot.loc[["train", "val", "test"]]
display(pivot)

### 2b. Load SBERT classifier and pre-computed feature matrices

The SBERT features were encoded in `train_models.ipynb` and saved as `.npy` files.
Loading them here is instant (no encoder required) so this notebook stays fast.

In [ ]:
# ADDED — load the SBERT Logistic Regression classifier from disk
sbert_logistic = load_object(os.path.join(MODELS_DIR, "sbert_logistic.pkl"))  # ADDED

# ADDED — load the pre-computed SBERT feature matrices (encoded in train_models.ipynb)
# These are plain numpy arrays: shape (n_samples, 768) = 2 * 384-dim MiniLM embeddings
X_train_sbert = np.load(os.path.join(MODELS_DIR, "sbert_X_train.npy"))  # ADDED
X_val_sbert   = np.load(os.path.join(MODELS_DIR, "sbert_X_val.npy"))    # ADDED
X_test_sbert  = np.load(os.path.join(MODELS_DIR, "sbert_X_test.npy"))   # ADDED

print("SBERT model and feature matrices loaded.")           # ADDED
print(f"  train shape: {X_train_sbert.shape}")            # ADDED
print(f"  val   shape: {X_val_sbert.shape}")              # ADDED
print(f"  test  shape: {X_test_sbert.shape}")             # ADDED

### 4b. SBERT model evaluation

The SBERT model uses `paraphrase-MiniLM-L6-v2` semantic embeddings
(`|emb_q1 − emb_q2|` and `emb_q1 ⊙ emb_q2`) fed into a LogisticRegression.
Unlike the lexical models above, SBERT captures synonym and paraphrase
relationships invisible to BoW / TF-IDF.

In [ ]:
# ADDED — evaluate the SBERT model on all three splits and append to results_df
sbert_rows = [  # ADDED
    evaluate_model(sbert_logistic, X_train_sbert, y_train, "sbert", "train"),  # ADDED
    evaluate_model(sbert_logistic, X_val_sbert,   y_val,   "sbert", "val"),    # ADDED
    evaluate_model(sbert_logistic, X_test_sbert,  y_test,  "sbert", "test"),   # ADDED
]  # ADDED

# Merge SBERT results into the main results table
sbert_df   = pd.DataFrame(sbert_rows).set_index(["model", "split"])  # ADDED
results_df = pd.concat([results_df, sbert_df])                        # ADDED
display(results_df)                                                    # ADDED

### 5b. Summary pivot — all three models side by side

In [ ]:
# ADDED — full three-model pivot: baseline | improved | sbert
full_pivot = results_df.reset_index().pivot(index="split", columns="model")  # ADDED
full_pivot = full_pivot.loc[["train", "val", "test"]]                         # ADDED
display(full_pivot)                                                            # ADDED